## Debris emissions and river discharge
Analyzing debris emissions and river discharge for seasonal distrubtion

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import glob
import os

In [ ]:

gdf = gpd.read_file(r"../ROI/northBob.shp")

print(gdf.head())

In [ ]:
sumofMtperyear = np.sum(gdf['dots_exten'])
print(sumofMtperyear)

### Area 1 Distribution based on river discharge 

In [ ]:
#1. Load Plastic Emissions Data (Meijer 2021)

print(f"Total Annual Emissions: {sumofMtperyear:.2f} Mt/year")


#### Area 1 fractions monthly

In [ ]:

sumofMtperyear=34296.918594

glofas_dir = r"../data/glofasrdischarge"

glofas_files = sorted(glob.glob(os.path.join(glofas_dir, "*.nc")))

print(f"Loading {len(glofas_files)} GLOFAS files...")
# Use open_mfdataset to handle multiple files
ds_glofas = xr.open_mfdataset(glofas_files, combine='nested', concat_dim='forecast_reference_time')
# Calculate monthly distribution for GLOFAS discharge
# We take the median across ensemble members (number), forecast period, and spatial dimensions
glofas_mean = ds_glofas["dis24"].median(dim=["number", "forecast_period", "latitude", "longitude"])
monthly_glofas = glofas_mean.groupby("forecast_reference_time.month").mean()

fractions2 = monthly_glofas / monthly_glofas.sum()

# 3. Apply Distribution to Annual Total
monthly_plastic2 = sumofMtperyear * fractions2

# 4. Generate Results Table
table2 = monthly_plastic2.to_dataframe(name="Tonnes of debris").reset_index()
table2["month_name"] = pd.to_datetime(table2["month"], format="%m").dt.strftime("%b")
table2 = table2[["month_name", "Tonnes of debris"]]


In [ ]:
df = glofas_mean.to_dataframe(name="discharge").reset_index()

df["year"] = df["forecast_reference_time"].dt.year
df["month"] = df["forecast_reference_time"].dt.month

monthly_stats = df.groupby("month")["discharge"].agg([
    "mean",
    "std",
    "min",
    "max"
])

print(monthly_stats)

In [ ]:

df.boxplot(column="discharge", by="month")
plt.yscale("log")
plt.xlabel("Month")
plt.ylabel("Discharge (m³/s)")
plt.title("Interannual discharge variability Area 1")
plt.suptitle("")
plt.show()

In [ ]:

print("\nSeasonal Plastic Emissions (based on GLOFAS):")
print(table2)


In [ ]:
months = np.arange(1,13)
month_labels=['Jan','Feb','Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# 5. Plot the results
plt.figure(figsize=(10, 6))
plt.yscale("log")
plt.plot(months, monthly_plastic2, marker='o', linestyle='-', color='red')
plt.xticks(months, month_labels)
plt.xlabel("Month")
plt.ylabel("Debris emissions (tonnes)")
plt.title("Seasonal Plastic Emissions – Northern Bay of Bengal (GloFAS river discharge 2020-2025)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


### Only Kolkota region

In [ ]:
area1 = gpd.read_file(r"../ROI/Kolkotaregion.shp")

print(area1.head())

In [ ]:
sources1 = area1['dots_exten'].values
coord = area1.get_coordinates()
print(coord)

In [ ]:
print(fractions2.values)

In [ ]:
fractions_df = fractions2.to_dataframe(name="fraction").reset_index()
fractions_df["month_name"] = pd.to_datetime(fractions_df["month"], format="%m").dt.strftime("%b")
fractions = fractions_df["fraction"].values

# Multiply (broadcasting)
result = sources1[:, None] * fractions *1000 #to get values in kg

# Build DataFrame
table_final = pd.DataFrame(result, columns=fractions_df["month_name"])
coord = coord.rename(columns={"x": "lon", "y": "lat"})#adjust if needed

# Reset index to align properly
table_final = table_final.reset_index(drop=True)
coord = coord.reset_index(drop=True)

# Add coordinates at the beginning
table_final = pd.concat([coord, table_final], axis=1)
print(table_final)
# --- Export ---
table_final.to_excel("monthly_sourcesarea1.xlsx", index=False)

### Area 2 discharge rates

In [ ]:
area2 = gpd.read_file(r"../ROI/midpointsarea2.geojson")

#print(area2.head())
eachsource =area2['dots_exten']
print(eachsource)
sumMtarea2= np.sum(area2['dots_exten'])
print(sumMtarea2)

#coordinates 17-18N, 83-84E

In [ ]:
glofas_dir2 = r"../data/glofasarea2"


In [ ]:
glofas_files2 = sorted(glob.glob(os.path.join(glofas_dir2, "*.nc")))

print(f"Loading {len(glofas_files2)} GLOFAS files...")
# Use open_mfdataset to handle multiple files
ds_glofas2 = xr.open_mfdataset(glofas_files2, combine='nested', concat_dim='forecast_reference_time')
# Calculate monthly distribution for GLOFAS discharge


#### Monthly fractions based on all years

In [ ]:
# We take the median across ensemble members (number), forecast period, and spatial dimensions
glofas_mean2 = ds_glofas2["dis24"].median(dim=["number", "forecast_period", "latitude", "longitude"])
monthly_glofas2 = glofas_mean2.groupby("forecast_reference_time.month").mean()

fractions_2 = monthly_glofas2 / monthly_glofas2.sum()

# 3. Apply Distribution to Annual Total
monthly_plastic21 = sumMtarea2 * fractions_2


In [ ]:

# 4. Generate Results Table
# First table: tonnes
table21 = monthly_plastic21.to_dataframe(name="Tonnes of debris").reset_index()
table21["month_name"] = pd.to_datetime(table21["month"], format="%m").dt.strftime("%b")

# Second table: fractions
fractions_df = fractions_2.to_dataframe(name="Fraction per month").reset_index()
fractions_df["month_name"] = pd.to_datetime(fractions_df["month"], format="%m").dt.strftime("%b")

# Merge both tables on month (or month_name)
table21 = table21.merge(fractions_df, on=["month", "month_name"])

# Select desired columns
table21 = table21[["month_name", "Tonnes of debris", "Fraction per month"]]

In [ ]:
print("\nSeasonal Plastic Emissions for Area 2 (based on GloFAS):")
print(table21)


In [ ]:

# Sources (12 rows)
sources = area2['dots_exten'].values
coord = area2.get_coordinates()
print(coord)


#### Area 2 sources to xlxs


In [ ]:

# Fractions (12 months)
fractions_df = fractions_2.to_dataframe(name="fraction").reset_index()
fractions_df["month_name"] = pd.to_datetime(fractions_df["month"], format="%m").dt.strftime("%b")
fractions = fractions_df["fraction"].values

# Multiply (broadcasting)
result = sources[:, None] * fractions * 1000 # get result in kg

# Build DataFrame
table_final = pd.DataFrame(result, columns=fractions_df["month_name"])
coord = coord.rename(columns={"x": "lon", "y": "lat"})#adjust if needed

# Reset index to align properly
table_final = table_final.reset_index(drop=True)
coord = coord.reset_index(drop=True)

# Add coordinates at the beginning
table_final = pd.concat([coord, table_final], axis=1)
print(table_final)
# --- Export ---
table_final.to_excel("monthly_sourcesarea2.xlsx", index=False)

In [ ]:
months = np.arange(1,13)
month_labels=['Jan','Feb','Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# 5. Plot the results
plt.figure(figsize=(10, 6))
plt.yscale("log")
plt.plot(months, monthly_plastic21, marker='o', linestyle='-', color='red')
plt.xticks(months, month_labels)
plt.xlabel("Month")
plt.ylabel("Debris emissions (tonnes)")
plt.title("Seasonal Plastic Emissions – Area 2 Bay of Bengal (GloFAS river discharge 2020-2025)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
df2 = glofas_mean2.to_dataframe(name="discharge").reset_index()

df2["year"] = df2["forecast_reference_time"].dt.year
df2["month"] = df2["forecast_reference_time"].dt.month

In [ ]:

df2.boxplot(column="discharge", by="month")
plt.yscale("log")
plt.xlabel("Month")
plt.ylabel("Discharge (m³/s)")
plt.title("Interannual discharge variability Area 2")
plt.suptitle("")
plt.show()